In [1]:
from pathlib import Path
from needle_bridge import NeedleBridgeConfig, build_needle_run_plan

# -------------------------
# CONFIG
# -------------------------
cfg = NeedleBridgeConfig(
    matches_index_csv=Path("/media/hello/Vault/Tribunals/_Matches/_matches_index.csv"),
    match_frequencies_csv=None,  # not used (aggregate file)
    ws_enhanced_csv=Path("/home/hello/Projects/Statements/output/Leonardo_WS_enhanced.csv"),
    y_inferred_json=Path("/home/hello/Projects/Statements/output/Y_inferred.json"),

    # These match your real data
    et_path_col="path",
    ws_row_id_col="X1",
    out_row_id_col="row_id",
    y_row_prefix="X1_",
    y_row_pad=4,

    # Recommended filters
    filter_ws_any_needle=True,
    filter_et_any_needle=True,
)

# -------------------------
# BUILD RUN PLAN
# -------------------------
out = build_needle_run_plan(cfg)

et_df = out["et_df"]
ws_df = out["ws_df"]
row_x_tests_df = out["row_x_tests_df"]
run_plan_df = out["run_plan_df"]

# -------------------------
# QUICK DIAGNOSTICS
# -------------------------
print("ET docs:", len(et_df))
print("WS rows:", len(ws_df))
print("Y x_tests rows:", len(row_x_tests_df))
print("Run plan rows:", len(run_plan_df))
print()

print("Unique WS rows in plan:", run_plan_df["row_id"].nunique())
print("Unique ET docs in plan:", run_plan_df["et_path"].nunique())
print()

run_plan_df.head(20)

ET docs: 5062
WS rows: 3
Y x_tests rows: 55
Run plan rows: 27540

Unique WS rows in plan: 3
Unique ET docs in plan: 2577



,row_id,y_row_id,et_path,matched_needle,match_mode,x_key,x_name,x_scope
0,11,X1_0011,/media/hello/Vault/Tribunals/ET_Cases/Ms_X_Ju_...,has__assumed_intention_regex,NEEDLE_OVERLAP,X1,Factual Basis for Allegations,INTERNAL_DISCIPLINARY_APPEAL
1,11,X1_0011,/media/hello/Vault/Tribunals/ET_Cases/Ms_X_Ju_...,has__assumed_intention_regex,NEEDLE_OVERLAP,X2,Proportionality of Dismissal,INTERNAL_DISCIPLINARY_APPEAL
2,11,X1_0011,/media/hello/Vault/Tribunals/ET_Cases/Ms_X_Ju_...,has__assumed_intention_regex,NEEDLE_OVERLAP,X3,Management Expectations,INTERNAL_DISCIPLINARY_APPEAL
3,11,X1_0011,/media/hello/Vault/Tribunals/ET_Cases/Ms_X_Ju_...,has__assumed_intention_regex,NEEDLE_OVERLAP,X4,Lack of Corrective Guidance,INTERNAL_DISCIPLINARY_APPEAL
4,11,X1_0011,/media/hello/Vault/Tribunals/ET_Cases/Ms_X_Ju_...,has__assumed_intention_regex,NEEDLE_OVERLAP,X5,Comparator Evidence,INTERNAL_DISCIPLINARY_APPEAL
5,11,X1_0011,/media/hello/Vault/Tribunals/ET_Cases/Mrs_S_Me...,has__assumed_intention_regex,NEEDLE_OVERLAP,X1,Factual Basis for Allegations,INTERNAL_DISCIPLINARY_APPEAL
6,11,X1_0011,/media/hello/Vault/Tribunals/ET_Cases/Mrs_S_Me...,has__assumed_intention_regex,NEEDLE_OVERLAP,X2,Proportionality of Dismissal,INTERNAL_DISCIPLINARY_APPEAL
7,11,X1_0011,/media/hello/Vault/Tribunals/ET_Cases/Mrs_S_Me...,has__assumed_intention_regex,NEEDLE_OVERLAP,X3,Management Expectations,INTERNAL_DISCIPLINARY_APPEAL
8,11,X1_0011,/media/hello/Vault/Tribunals/ET_Cases/Mrs_S_Me...,has__assumed_intention_regex,NEEDLE_OVERLAP,X4,Lack of Corrective Guidance,INTERNAL_DISCIPLINARY_APPEAL
9,11,X1_0011,/media/hello/Vault/Tribunals/ET_Cases/Mrs_S_Me...,has__assumed_intention_regex,NEEDLE_OVERLAP,X5,Comparator Evidence,INTERNAL_DISCIPLINARY_APPEAL


In [ ]:
import pandas as pd
cols = ["row_id", "y_row_id"] + list(cfg.needle_cols)
print(ws_df[cols].to_string(index=False))
selector = [c for c in cfg.needle_cols if c != "has__any_needle"]
ws_active_counts = (ws_df[selector] == 1).sum(axis=1)
print(pd.DataFrame({
    "row_id": ws_df["row_id"],
    "y_row_id": ws_df["y_row_id"],
    "active_selector_needles": ws_active_counts
}).to_string(index=False))

In [ ]:
import sys, json, importlib, time
from pathlib import Path
from pprint import pprint
from tqdm.auto import tqdm

# ==========================================================
# MOLTIE RUNNER (INTEGRATED)
#  - consumes run_plan_df from the bridge
#  - DEBUG mode can select iloc rows + force single PDF
#  - batch mode runs plan as-is (optional caps)
# ==========================================================

# -------------------------
# REQUIRE: run_plan_df exists
# -------------------------
assert "run_plan_df" in globals(), "Missing run_plan_df. Build it with the Needle Bridge first."

required_cols = {"row_id", "y_row_id", "et_path", "x_key", "x_name"}
missing = required_cols - set(run_plan_df.columns)
assert not missing, f"run_plan_df missing columns: {missing}"

# -------------------------
# USER CONTROLS
# -------------------------
DEBUG = False

# DEBUG selection: can be int, list[int], or slice
PLAN_ILOC = [3]           # examples: 3, [3], [2,5,9], slice(0,3)

# Force single PDF in DEBUG mode
FORCE_PDF = Path("/home/hello/Projects/Statements/code/appeals/1._Mr_G_Lepiarz__2._Mr_D_Lewis_v__Trades_Union_Congress_-_2200228-2023___2200230-2023.pdf")

# Batch caps (only used when DEBUG=False)
ONLY_X_KEY = None                 # e.g. "X3"
MAX_DOCS_PER_YROW_AND_X = None    # e.g. 5
MAX_RUNS = None                   # e.g. 20

# Output
REPO_ROOT = Path("/home/hello/Projects/Statements").resolve()
OUT_DIR = REPO_ROOT / "output" / "moltie_batch"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Repo / code path / Y path
# -------------------------
CODE_ROOT = (REPO_ROOT / "code").resolve()
Y_PATH = REPO_ROOT / "output" / "Y_inferred.json"

assert CODE_ROOT.exists(), f"Missing CODE_ROOT: {CODE_ROOT}"
assert Y_PATH.exists(), f"Missing Y: {Y_PATH}"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

# -------------------------
# Imports (reload once)
# -------------------------
import moltie.schemas.run_config as rc_mod
import moltie.llm.verifier_prompt as vp_mod
import moltie.llm.client as client_mod
import moltie.schemas.query_object as qo_mod
import moltie.agent.loop as loop_mod

importlib.reload(rc_mod)
importlib.reload(vp_mod)
importlib.reload(client_mod)
importlib.reload(qo_mod)
importlib.reload(loop_mod)

RunConfig = rc_mod.RunConfig
AtomQuery = qo_mod.AtomQuery
run_agent_on_one_doc = loop_mod.run_agent_on_one_doc
LLMClientConfig = client_mod.LLMClientConfig

# -------------------------
# Load Y once
# -------------------------
y_root = json.loads(Y_PATH.read_text(encoding="utf-8"))
rows = y_root.get("rows") or {}
assert isinstance(rows, dict) and rows, "Y has no rows"

# -------------------------
# Build plan (DEBUG override vs batch)
# -------------------------
if DEBUG:
    assert FORCE_PDF.exists(), f"Forced PDF missing: {FORCE_PDF}"

    if isinstance(PLAN_ILOC, slice):
        plan = run_plan_df.iloc[PLAN_ILOC].copy()
    elif isinstance(PLAN_ILOC, list):
        plan = run_plan_df.iloc[PLAN_ILOC].copy()
    elif isinstance(PLAN_ILOC, int):
        plan = run_plan_df.iloc[[PLAN_ILOC]].copy()
    else:
        raise ValueError("PLAN_ILOC must be int, list, or slice")

    plan["et_path"] = str(FORCE_PDF)
else:
    plan = run_plan_df.copy()

    if ONLY_X_KEY is not None:
        plan = plan[plan["x_key"] == ONLY_X_KEY].copy()

    if MAX_DOCS_PER_YROW_AND_X is not None:
        plan = (
            plan.sort_values(["y_row_id", "x_key", "et_path"])
                .groupby(["y_row_id", "x_key"], as_index=False)
                .head(int(MAX_DOCS_PER_YROW_AND_X))
        )

    if MAX_RUNS is not None:
        plan = plan.head(int(MAX_RUNS)).copy()

plan = plan.reset_index(drop=True)

print("DEBUG:", DEBUG)
print("Plan rows:", len(plan))
print("Unique y_row_id:", plan["y_row_id"].nunique(), "| unique PDFs:", plan["et_path"].nunique())
print(plan[["row_id","y_row_id","x_key","x_name","et_path"]].head(10).to_string(index=False))

# -------------------------
# Client cfg (pinned once)
# -------------------------
_client_kwargs = dict(
    model="mistral-small3.2:latest",
    ollama_url="http://localhost:11434/api/generate",
    timeout_s=180,
    temperature=0.0,
    num_predict=1000,
    max_retries=2,
)

# optional stop/debug fields
try:
    if "stop" in getattr(LLMClientConfig, "__annotations__", {}):
        _client_kwargs["stop"] = []
except Exception:
    pass

try:
    if "debug" in getattr(LLMClientConfig, "__annotations__", {}):
        _client_kwargs["debug"] = DEBUG
except Exception:
    pass

client_cfg = LLMClientConfig(**_client_kwargs)

# -------------------------
# RunConfig (pinned once)
# -------------------------
cfg2 = RunConfig.from_dict({
    "debug": DEBUG,
    "harvest_mode": False,
    "max_iters": 3,
    "window_size": 12,
    "stride": 6,
    "top_windows": 3,
    "k_chunks_per_doc": 12,
    "anchors_required": 1,
    "min_hits": 1,
    "thresh_score": 1,
    "thresh_conf": 0.8,
    "plateau_p": 2,
    "eps_improve": 0,
    "iter_temp_enabled": True,
    "iter_temp_start": 0.20,
    "iter_temp_end": 0.00,
    "iter_temp_curve": "linear",
    "iter_temp_cap": 0.80,
})

# -------------------------
# PDF -> paras cache
# -------------------------
paras_cache = {}  # pdf_path_str -> paras(list[dict])

def get_paras(pdf_path: Path):
    k = str(pdf_path)
    if k in paras_cache:
        return paras_cache[k]

    from pypdf import PdfReader
    reader = PdfReader(str(pdf_path))
    text = "\n".join([(p.extract_text() or "") for p in reader.pages]).strip()
    paras = [{"para_id": "p00001", "text": text}]
    paras = loop_mod._maybe_rechunk_single_blob_paras(paras)

    paras_cache[k] = paras
    return paras

# -------------------------
# Output JSONL
# -------------------------
ts = time.strftime("%Y%m%d_%H%M%S")
OUT_JSONL = OUT_DIR / f"batch_results_{'DEBUG' if DEBUG else 'BATCH'}_{ts}.jsonl"
print("\nWriting results to:", OUT_JSONL)

def safe_to_dict(obj):
    if obj is None:
        return None
    if hasattr(obj, "to_dict"):
        return obj.to_dict()
    if isinstance(obj, dict):
        return obj
    return {"repr": repr(obj)}

n_ok = 0
n_neg = 0
n_err = 0

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for i, r in tqdm(plan.iterrows(), total=len(plan), desc="moltie run", unit="run"):
        y_row_id = r["y_row_id"]
        pdf_path = Path(r["et_path"])
        x_key = r["x_key"]
        x_name = r.get("x_name", x_key)

        try:
            if y_row_id not in rows:
                raise KeyError(f"y_row_id not in Y.rows: {y_row_id}")
            if not pdf_path.exists():
                raise FileNotFoundError(f"PDF missing: {pdf_path}")

            # row-scoped y object
            y_obj = (rows[y_row_id] or {}).get("y") or {}

            # parse / rechunk
            paras = get_paras(pdf_path)
            doc_id = pdf_path.stem

            # build atom
            merged = qo_mod.merge_indicators_and_excludes(y_obj, [x_key])
            atom = AtomQuery(
                atom_id=x_key,
                x_tests=[x_key],
                proposition=x_name,
                positive_indicators=merged["positive_indicators"],
                excludes=merged["excludes"],
                keyword_seeds=merged["positive_indicators"],
                expansion_terms=[],
            )

            # run
            res = run_agent_on_one_doc(doc_id, paras, atom, cfg2, client_cfg)

            verdict = safe_to_dict(getattr(res, "verdict", None))
            negative_exit = safe_to_dict(getattr(res, "negative_exit", None))

            if verdict:
                n_ok += 1
            else:
                n_neg += 1

            row_out = {
                "i": int(i),
                "row_id": r.get("row_id"),
                "y_row_id": y_row_id,
                "needle_signature": r.get("needle_signature"),
                "et_path": str(pdf_path),
                "doc_id": doc_id,
                "x_key": x_key,
                "x_name": x_name,
                "verdict": verdict,
                "negative_exit": negative_exit,
                "iters": getattr(res, "iters", None),
                "trace_tail": (getattr(res, "trace", None) or [])[-3:],
            }

            f.write(json.dumps(row_out, ensure_ascii=False) + "\n")

            if DEBUG:
                print("\n--- RUN", i, "---")
                print("y_row_id:", y_row_id, "| x_key:", x_key, "| pdf:", pdf_path.name)
                if verdict:
                    print("relevant=", verdict.get("relevant"),
                          "score=", verdict.get("precedent_score"),
                          "conf=", verdict.get("confidence"),
                          "anchors=", len(verdict.get("anchors") or []))
                else:
                    print("NEGATIVE:", (negative_exit or {}).get("reason"))

        except Exception as e:
            n_err += 1
            f.write(json.dumps({
                "i": int(i),
                "row_id": r.get("row_id"),
                "y_row_id": y_row_id,
                "et_path": str(pdf_path),
                "x_key": x_key,
                "error": repr(e),
            }, ensure_ascii=False) + "\n")

print("\nDONE")
print("ok:", n_ok, "| negative:", n_neg, "| errors:", n_err)
print("results:", OUT_JSONL)

In [ ]:
from pathlib import Path
import json
import pandas as pd

# -------------------------
# INPUT: point to your JSONL file
# -------------------------
#JSONL_PATH = Path("/home/hello/Projects/Statements/output/moltie_batch").glob("*.jsonl")
#JSONL_PATH = sorted(JSONL_PATH)[-1]  # latest file



JSONL_PATH = Path("/home/hello/Projects/Statements/output/moltie_batch/batch_results_BATCH_20260225_172948.jsonl")

print("Loading:", JSONL_PATH)

# -------------------------
# LOAD JSONL
# -------------------------
rows = []
with JSONL_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

print("Raw shape:", df.shape)

# -------------------------
# FLATTEN verdict fields
# -------------------------
def extract_field(d, key):
    if isinstance(d, dict):
        return d.get(key)
    return None

df["relevant"] = df["verdict"].apply(lambda v: extract_field(v, "relevant"))
df["precedent_score"] = df["verdict"].apply(lambda v: extract_field(v, "precedent_score"))
df["confidence"] = df["verdict"].apply(lambda v: extract_field(v, "confidence"))
df["matched_X"] = df["verdict"].apply(lambda v: extract_field(v, "matched_X"))

df["anchor_count"] = df["verdict"].apply(
    lambda v: len(v.get("anchors") or []) if isinstance(v, dict) else 0
)

# -------------------------
# SELECT CLEAN COLUMNS
# -------------------------
final_cols = [
    "row_id",
    "y_row_id",
    "x_key",
    "x_name",
    "et_path",
    "relevant",
    "precedent_score",
    "confidence",
    "anchor_count",
]

df_out = df[final_cols].copy()

# -------------------------
# SAVE CSV
# -------------------------
CSV_PATH = JSONL_PATH.with_suffix(".csv")
df_out.to_csv(CSV_PATH, index=False)

print("Saved CSV:", CSV_PATH)
print("Final shape:", df_out.shape)

df_out.head(10)

In [ ]:
df_true = df_out[df_out["relevant"] == True].copy()
print((df_true))

In [ ]:
path = r"/media/hello/Vault/Tribunals/_Matches/_matches_index.csv"

matches_df = pd.read_csv(path)
total_rows = len(matches_df)
unique_paths = matches_df["path"].nunique()

print("Total rows:", total_rows)
print("Unique paths:", unique_paths)
print("Are all files unique?", total_rows == unique_paths)